# AML Pipeline - Bloco de Inicialização Completo

## ✅ Objetivo

Este notebook demonstra como usar o bloco de inicialização para ativar todas as 5 ETAPAs refatoradas:

- **ETAPA 1-2**: Feature Engineering (Velocity, Ratio, Behavioral)
- **ETAPA 3**: Bayesian Optimization com Optuna
- **ETAPA 4**: Transformações com Anti-Leakage (YeoJohnson, TargetEncoder)
- **ETAPA 5**: Métricas de Negócio (Precision@K, PSI)

**Instruções**: Execute todas as células na ordem apresentada para validar o ambiente.
___

## 1️⃣ Configuração Dinâmica de Path

Detecta o diretório raiz do projeto e adiciona ao sys.path para permitir importações de `source/`

In [ ]:
import sys
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

print("="*100)
print("INICIALIZANDO AMBIENTE AML - NOTEBOOK SETUP")
print("="*100)

# ============================================================================
# 1. AJUSTE DE PATH DINÂMICO
# ============================================================================
print("\n[1/5] Configurando Path Dinâmico...")

# Se estamos em um notebook, usar o working directory
notebook_dir = Path.cwd()
if (notebook_dir / "source").exists():
    proj_root = notebook_dir
elif (notebook_dir.parent / "source").exists():
    proj_root = notebook_dir.parent
elif (notebook_dir.parent.parent / "source").exists():
    proj_root = notebook_dir.parent.parent
else:
    # Fallback: procurar por pyproject.toml
    for p in [notebook_dir, notebook_dir.parent, notebook_dir.parent.parent]:
        if (p / "pyproject.toml").exists():
            proj_root = p
            break
    else:
        raise FileNotFoundError(
            "⚠ Não foi possível localizar o diretório raiz do projeto.\n"
            "Verifique se está na pasta notebooks/ e se source/ existe."
        )

# Adicionar ao sys.path se não estiver
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))
    print(f"  ✓ Path adicionado: {proj_root}")
else:
    print(f"  ✓ Path já configurado: {proj_root}")

print(f"  Current Working Directory: {notebook_dir}")

## 2️⃣ Configuração Spark para AML

Inicializa SparkSession otimizado para processamento de features de alta dimensionalidade

In [ ]:
# ============================================================================
# 2. CONFIGURAÇÃO SPARK PARA AML
# ============================================================================
print("\n[2/5] Configurando SparkSession para AML...")

try:
    from pyspark.sql import SparkSession
    from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
    
    # Configurar SparkSession com otimizações para AML
    spark = SparkSession.builder \
        .appName("AML_Detection_Pipeline") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .config("spark.sql.shuffle.partitions", "100") \
        .config("spark.driver.memory", "4g") \
        .config("spark.executor.memory", "4g") \
        .config("spark.memory.fraction", "0.8") \
        .config("spark.sql.broadcastTimeout", "360") \
        .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
        .getOrCreate()
    
    spark.sparkContext.setLogLevel("ERROR")
    print(f"  ✓ SparkSession inicializada")
    print(f"    - App Name: {spark.appName}")
    print(f"    - Driver Memory: 4g")
    print(f"    - Executor Memory: 4g")
    
except ImportError:
    print("  ⚠ PySpark não disponível - Esta é uma sessão sem Spark")
    print("    (Continuando com operações locais em Pandas/NumPy)")
    spark = None
except Exception as e:
    print(f"  ⚠ Erro ao inicializar Spark: {e}")
    spark = None

## 3️⃣ Importação dos Módulos Refatorados

Carrega todas as classes e funções das 5 ETAPAs implementadas

In [ ]:
# ============================================================================
# 3. IMPORTAÇÃO DOS MÓDULOS REFATORADOS
# ============================================================================
print("\n[3/5] Importando módulos refatorados...")

# Dicionário para rastrear importações bem-sucedidas
imports_status = {}

# ETAPA 1-2: Features e Preprocessing
try:
    from source.config import PROJ_ROOT, PROCESSED_DATA_DIR, MODELS_DIR
    imports_status['source.config'] = True
    print("  ✓ source.config")
except ImportError as e:
    imports_status['source.config'] = False
    print(f"  ✗ source.config: {e}")

try:
    from source.features import (
        FeatureEngineer,
        VelocityFeature,
        RatioFeature,
        BehavioralFeature
    )
    imports_status['source.features'] = True
    print("  ✓ source.features (Feature Engineering)")
except ImportError as e:
    imports_status['source.features'] = False
    print(f"  ✗ source.features: {e}")

# ETAPA 4: Preprocessing com Anti-Leakage
try:
    from source.preprocessing import (
        AMLPreprocessor,
        YeoJohnsonTransformerSafe,
        TargetEncoderRegularized,
        FrequencyEncoder,
        DateTimeFeatureExtractor
    )
    imports_status['source.preprocessing'] = True
    print("  ✓ source.preprocessing (ETAPA 4: Anti-Leakage)")
except ImportError as e:
    imports_status['source.preprocessing'] = False
    print(f"  ✗ source.preprocessing: {e}")

# ETAPA 3: Bayesian Optimization e Anomaly Detection
try:
    from source.modeling.optuna_tuner import AMLTunerPipeline
    imports_status['source.modeling.optuna_tuner'] = True
    print("  ✓ source.modeling.optuna_tuner (ETAPA 3: Bayesian Optimization)")
except ImportError as e:
    imports_status['source.modeling.optuna_tuner'] = False
    print(f"  ✗ source.modeling.optuna_tuner: {e}")

# ETAPA 5: Métricas de Negócio
try:
    from source.modeling.metrics import (
        BusinessMetrics,
        ModelEvaluationReport
    )
    imports_status['source.modeling.metrics'] = True
    print("  ✓ source.modeling.metrics (ETAPA 5: Precision@K e PSI)")
except ImportError as e:
    imports_status['source.modeling.metrics'] = False
    print(f"  ✗ source.modeling.metrics: {e}")

# Utilitários
try:
    from source.dataset import DataLoader
    imports_status['source.dataset'] = True
    print("  ✓ source.dataset")
except ImportError as e:
    imports_status['source.dataset'] = False
    print(f"  ✗ source.dataset: {e}")

try:
    from source.plots import ModelPlotter
    imports_status['source.plots'] = True
    print("  ✓ source.plots")
except ImportError as e:
    imports_status['source.plots'] = False
    print(f"  ✗ source.plots: {e}")

## 4️⃣ Verificação de Dependências

Valida a disponibilidade de todas as bibliotecas críticas

In [ ]:
# ============================================================================
# 4. VERIFICAÇÃO DE VERSÃO E DEPENDÊNCIAS
# ============================================================================
print("\n[4/5] Verificando dependências críticas...")

dependencies_status = {}

# Verificar Python
try:
    import platform
    python_version = f"{platform.python_version()}"
    dependencies_status['Python'] = (python_version, True)
    print(f"  ✓ Python {python_version}")
except Exception as e:
    dependencies_status['Python'] = (None, False)
    print(f"  ✗ Python: {e}")

# Verificar NumPy
try:
    import numpy as np
    np_version = np.__version__
    dependencies_status['NumPy'] = (np_version, True)
    print(f"  ✓ NumPy {np_version}")
except ImportError:
    dependencies_status['NumPy'] = (None, False)
    print(f"  ✗ NumPy: não disponível")

# Verificar Pandas
try:
    import pandas as pd
    pd_version = pd.__version__
    dependencies_status['Pandas'] = (pd_version, True)
    print(f"  ✓ Pandas {pd_version}")
except ImportError:
    dependencies_status['Pandas'] = (None, False)
    print(f"  ✗ Pandas: não disponível")

# Verificar scikit-learn
try:
    import sklearn
    sklearn_version = sklearn.__version__
    dependencies_status['scikit-learn'] = (sklearn_version, True)
    print(f"  ✓ scikit-learn {sklearn_version}")
except ImportError:
    dependencies_status['scikit-learn'] = (None, False)
    print(f"  ✗ scikit-learn: não disponível")

# Verificar XGBoost
try:
    import xgboost as xgb
    xgb_version = xgb.__version__
    dependencies_status['XGBoost'] = (xgb_version, True)
    print(f"  ✓ XGBoost {xgb_version}")
except ImportError:
    dependencies_status['XGBoost'] = (None, False)
    print(f"  ✗ XGBoost: não disponível")

# Verificar LightGBM
try:
    import lightgbm as lgb
    lgb_version = lgb.__version__
    dependencies_status['LightGBM'] = (lgb_version, True)
    print(f"  ✓ LightGBM {lgb_version}")
except ImportError:
    dependencies_status['LightGBM'] = (None, False)
    print(f"  ✗ LightGBM: não disponível")

# Verificar Optuna (ETAPA 3)
try:
    import optuna
    optuna_version = optuna.__version__
    dependencies_status['Optuna'] = (optuna_version, True)
    print(f"  ✓ Optuna {optuna_version} (ETAPA 3: Bayesian Optimization)")
except ImportError:
    dependencies_status['Optuna'] = (None, False)
    print(f"  ✗ Optuna: não disponível")

# Verificar Loguru
try:
    from loguru import logger
    dependencies_status['Loguru'] = ('available', True)
    print(f"  ✓ Loguru (logging com structured output)")
except ImportError:
    dependencies_status['Loguru'] = (None, False)
    print(f"  ✗ Loguru: não disponível")

## 5️⃣ Sumário e Validação de Ambiente

Gera relatório consolidado de inicialização

In [ ]:
# ============================================================================
# 5. SUMÁRIO E INFORMAÇÕES DE AMBIENTE
# ============================================================================
print("\n[5/5] Sumário de Ambiente")
print("-" * 100)

# Status geral
all_imports_ok = all(imports_status.values())
all_deps_ok = all(status[1] for status in dependencies_status.values())

if all_imports_ok:
    print("✓ TODOS OS MÓDULOS IMPORTADOS COM SUCESSO")
else:
    failed = [k for k, v in imports_status.items() if not v]
    print(f"⚠ PROBLEMAS COM IMPORTAÇÃO: {', '.join(failed)}")

if all_deps_ok:
    print("✓ TODAS AS DEPENDÊNCIAS DISPONÍVEIS")
else:
    failed = [k for k, (v, ok) in dependencies_status.items() if not ok]
    print(f"⚠ DEPENDÊNCIAS FALTANDO: {', '.join(failed)}")

print("-" * 100)

# Informações do Projeto
print("\nINFORMAÇÕES DO PROJETO:")
print(f"  Diretório Raiz: {proj_root}")
print(f"  Working Directory: {Path.cwd()}")
try:
    print(f"  Processed Data: {PROCESSED_DATA_DIR}")
    print(f"  Models Directory: {MODELS_DIR}")
except:
    print(f"  Processed Data: N/A")
    print(f"  Models Directory: N/A")

# Informações de Spark
if spark:
    print(f"\nINFORMAÇÕES SPARK:")
    print(f"  Status: Ativo")
    print(f"  App Name: {spark.appName}")
    print(f"  Master: {spark.sparkContext.master}")
else:
    print(f"\nINFORMAÇÕES SPARK:")
    print(f"  Status: Não disponível")

# Informações de ETAPAS
print("\nETAPAS IMPLEMENTADAS:")
print(f"  ETAPA 3 (Tuning + Anomaly): {'✓' if imports_status.get('source.modeling.optuna_tuner', False) else '✗'}")
print(f"  ETAPA 4 (Anti-Leakage): {'✓' if imports_status.get('source.preprocessing', False) else '✗'}")
print(f"  ETAPA 5 (Métricas AML): {'✓' if imports_status.get('source.modeling.metrics', False) else '✗'}")

print("\n" + "="*100)
print("AMBIENTE PRONTO! Você pode começar a trabalhar com os dados.")
print("="*100 + "\n")

## 6️⃣ Funções Auxiliares Úteis

Defina funções helper para uso rápido em análises posteriores

In [ ]:
# ============================================================================
# Funções auxiliares úteis
# ============================================================================

def show_status():
    """Exibe status resumido de todas as importações."""
    print("\nSTATUS DE IMPORTAÇÕES:")
    for module, status in imports_status.items():
        symbol = "✓" if status else "✗"
        print(f"  [{symbol}] {module}")
    
    print("\nDEPENDÊNCIAS:")
    for dep, (version, status) in dependencies_status.items():
        symbol = "✓" if status else "✗"
        ver_str = f"{version}" if version else "N/A"
        print(f"  [{symbol}] {dep:<20} → {ver_str}")

def load_and_prepare_data(data_type='train', sample_size=None):
    """
    Helper para carregar dados de treino ou OOT.
    
    Args:
        data_type: 'train' ou 'oot'
        sample_size: Número de linhas para amostrar (None = todos)
    
    Returns:
        DataFrame com os dados
    """
    try:
        import pandas as pd
        
        if data_type == 'train':
            path = PROCESSED_DATA_DIR / 'df_treino.csv'
        else:
            path = PROCESSED_DATA_DIR / 'df_oot.csv'
        
        df = pd.read_csv(path)
        
        if sample_size and sample_size < len(df):
            df = df.sample(n=sample_size, random_state=42)
        
        print(f"✓ Dados carregados: {path.name} ({df.shape[0]} linhas, {df.shape[1]} colunas)")
        return df
    
    except Exception as e:
        print(f"✗ Erro ao carregar dados: {e}")
        return None

def validate_etapa_implementations():
    """Valida se todas as ETAPAs foram implementadas corretamente."""
    print("\nVALIDAÇÃO DE ETAPAS:")
    
    # ETAPA 3: Optuna + Isolation Forest
    try:
        from source.modeling.optuna_tuner import AMLTunerPipeline
        print("  ✓ ETAPA 3: AMLTunerPipeline com Bayesian Optimization")
    except:
        print("  ✗ ETAPA 3: AMLTunerPipeline não encontrada")
    
    # ETAPA 4: YeoJohnson + TargetEncoder
    try:
        from source.preprocessing import YeoJohnsonTransformerSafe, TargetEncoderRegularized
        print("  ✓ ETAPA 4: YeoJohnsonTransformerSafe + TargetEncoderRegularized")
    except:
        print("  ✗ ETAPA 4: Transformadores anti-leakage não encontrados")
    
    # ETAPA 5: Precision@K + PSI
    try:
        from source.modeling.metrics import BusinessMetrics, ModelEvaluationReport
        
        # Testar métodos
        assert hasattr(BusinessMetrics, 'precision_at_top_k')
        assert hasattr(BusinessMetrics, 'calculate_psi')
        assert hasattr(ModelEvaluationReport, 'add_model_metrics')
        
        print("  ✓ ETAPA 5: Precision@Top-K + PSI + ModelEvaluationReport")
    except:
        print("  ✗ ETAPA 5: Métricas de negócio não encontradas")

# Validar ETAPAS automaticamente
validate_etapa_implementations()

print("\n💡 DICAS:")
print("  • Use show_status() para ver status de importações")
print("  • Use load_and_prepare_data('train') para carregar dados de treino")
print("  • Use load_and_prepare_data('oot') para carregar dados de OOT")
print("  • Variáveis spark, pd, np estão disponíveis")
print("  • Métricas de ETAPA 5 prontas: BusinessMetrics.precision_at_top_k(), .calculate_psi()")

---

# Exemplos de Uso das ETAPAs

Nas células abaixo temos exemplos práticos de como usar cada uma das ETAPAs refatoradas.

## Exemplo 1: Carregar Dados e Aplicar Transformações (ETAPA 4)

Demonstra o uso do YeoJohnsonTransformerSafe com garantia de anti-leakage

In [ ]:
# Carregar dados de treino
df_train = load_and_prepare_data('train', sample_size=5000)

if df_train is not None:
    print(f"\nDados carregados:")
    print(f"  Forma: {df_train.shape}")
    print(f"  Colunas numéricas: {df_train.select_dtypes(include=['number']).shape[1]}")
    print(f"\nPrimeiras linhas:")
    print(df_train.head())

## Exemplo 2: Aplicar Transformação com Anti-Leakage (ETAPA 4)

Usa YeoJohnsonTransformerSafe garantindo que lambda é calculada APENAS no fit()

In [ ]:
from source.preprocessing import YeoJohnsonTransformerSafe
import numpy as np

# Selecionar coluna numérica para exemplo
if df_train is not None and 'amount' in df_train.columns:
    X_train = df_train[['amount']].values
    
    # Criar e aplicar transformador
    transformer = YeoJohnsonTransformerSafe()
    
    # FIT: Calcula lambda
    X_train_transformed = transformer.fit_transform(X_train)
    
    print("ETAPA 4 - Transformação com Anti-Leakage:")
    print(f"  ✓ Lambda calculada NO FIT (anti-leakage garantido)")
    print(f"  Dados antes: mean={X_train.mean():.4f}, std={X_train.std():.4f}")
    print(f"  Dados depois: mean={X_train_transformed.mean():.4f}, std={X_train_transformed.std():.4f}")
    print(f"\n  Este transformador pode ser seguramente usado em pipeline de treino/teste!")
else:
    print("  ⚠ Dados não carregados. Pule para exemplo anterior.")

## Exemplo 3: Calcular Métricas de Precisão@Top-K (ETAPA 5)

Demonstra a métrica de negócio Precision@Top-K para ranking de detecção

In [ ]:
from source.modeling.metrics import BusinessMetrics

# Simular scores de predição (probabilidades)
np.random.seed(42)
y_true = np.random.binomial(1, 0.05, 1000)  # 5% positivos (desbalanceado)
y_scores = np.random.uniform(0, 1, 1000)

# Calcular Precision@Top-K
top_k_values = [100, 500]
precision_at_k = BusinessMetrics.precision_at_top_k(y_true, y_scores, top_k_list=top_k_values)

print("ETAPA 5 - Métricas de Negócio:")
print(f"\nPrecision@Top-K (para detecção focus de fraude):")
for k, precision in precision_at_k.items():
    print(f"  Precision@{k}: {precision:.4f}")
    
print(f"\nInterpretação:")
print(f"  - Precision@100: Qual proporção dos top-100 suspeitos é realmente fraude?")
print(f"  - Precision@500: Qual proporção dos top-500 suspeitos é realmente fraude?")

## Exemplo 4: Calcular PSI - Population Stability Index (ETAPA 5)

Monitora degradação do modelo detectando mudanças distributivas

In [ ]:
from source.modeling.metrics import BusinessMetrics

# Simular scores de treino e OOT (out-of-time)
np.random.seed(42)
train_scores = np.random.normal(0.5, 0.15, 1000)  # Treino
oot_scores = np.random.normal(0.52, 0.16, 1000)   # OOT com pequena degradação

# Calcular PSI
psi_value = BusinessMetrics.calculate_psi(train_scores, oot_scores, n_bins=10)

print("ETAPA 5 - PSI (Population Stability Index):")
print(f"\nPSI = {psi_value:.4f}")

if psi_value < 0.1:
    status = "✓ ESTÁVEL"
    interpretation = "Modelo comporta-se consistentemente em OOT"
elif psi_value < 0.25:
    status = "⚠ DEGRADAÇÃO MODERADA"
    interpretation = "Mudanças detectadas, mas ainda aceitável"
else:
    status = "✗ DEGRADAÇÃO SEVERA"
    interpretation = "Modelo não generaliza bem, retrainamento necessário"

print(f"Status: {status}")
print(f"Interpretação: {interpretation}")
print(f"\nReferência:")
print(f"  PSI < 0.1    → Estável")
print(f"  0.1 < PSI < 0.25 → Moderada")
print(f"  PSI > 0.25   → Severa")

## Exemplo 5: Relatório Consolidado de Avaliação (ETAPA 5)

Cria relatório unified de todas as métricas (tradicionais + negócio + PSI)

In [ ]:
from source.modeling.metrics import ModelEvaluationReport
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Simular predições de dois modelos
np.random.seed(42)

# Dados treino
y_train = np.random.binomial(1, 0.05, 1000)
y_train_pred_xgb = np.random.uniform(0, 1, 1000)
y_train_pred_lgb = np.random.uniform(0, 1, 1000)

# Dados OOT
y_oot = np.random.binomial(1, 0.05, 500)
y_oot_pred_xgb = np.random.uniform(0, 1, 500)
y_oot_pred_lgb = np.random.uniform(0, 1, 500)

# Criar relatório
report = ModelEvaluationReport()

# Calcular métricas tradicionais
for model_name, y_train_scores, y_oot_scores in [
    ('XGBoost_v1', y_train_pred_xgb, y_oot_pred_xgb),
    ('LightGBM_v1', y_train_pred_lgb, y_oot_pred_lgb)
]:
    standard_metrics = {
        'accuracy': accuracy_score(y_train, (y_train_scores > 0.5).astype(int)),
        'precision': precision_score(y_train, (y_train_scores > 0.5).astype(int), zero_division=0),
        'recall': recall_score(y_train, (y_train_scores > 0.5).astype(int), zero_division=0),
        'f1': f1_score(y_train, (y_train_scores > 0.5).astype(int), zero_division=0)
    }
    
    # Adicionar ao relatório
    report.add_model_metrics(
        model_name=model_name,
        y_true_train=y_train,
        y_scores_train=y_train_scores,
        y_true_oot=y_oot,
        y_scores_oot=y_oot_scores,
        standard_metrics=standard_metrics,
        top_k_list=[100, 500]
    )

# Gerar e exibir relatório
print("ETAPA 5 - Relatório Consolidado de Avaliação:\n")
report.print_summary()

# Obter como DataFrame
df_report = report.generate_report()
print("\nDataFrame do Relatório:")
print(df_report)

---

## ✅ Próximos Passos

Agora que o ambiente está inicializado com sucesso:

1. **Dados**: Use `load_and_prepare_data()` para carregar seu dataset
2. **Feature Engineering**: Aplique funções de `source.features` para criar Velocity, Ratio e Behavioral features
3. **Transformação**: Use `YeoJohnsonTransformerSafe` e `TargetEncoderRegularized` sem risco de data leakage
4. **Otimização**: Aplique `AMLTunerPipeline` com Optuna para Bayesian Optimization de hiperparâmetros
5. **Avaliação**: Use `BusinessMetrics` e `ModelEvaluationReport` para validar Precision@K e PSI

**Referência**: Consulte os arquivos de documentação:
- `NOTEBOOK_INITIALIZATION_GUIDE.md` - Guia completo de inicialização
- `ETAPA_3_COMPLETADA.md` - Documentação técnica ETAPA 3
- `ETAPA_4_COMPLETADA.md` - Documentação técnica ETAPA 4  
- `ETAPA_5_COMPLETADA.md` - Documentação técnica ETAPA 5